# Data Cleaning & Standardisation

> **Purpose**: Demonstrate the `cleaning.py` module — load the raw CSV and run the full cleaning pipeline.

**All business logic lives in `../cleaning.py`. This notebook only imports and calls those functions.**

### What happens in this stage
| Step | Function | What it does |
|------|----------|--------------|
| 1 | `load_dataset()` | Read raw CSV from disk |
| 2 | `clean_dates()` | Parse Transaction_Date → datetime (coerce bad values to NaT) |
| 3 | `parse_prices()` | Strip currency symbols from Price → float |
| 4 | `standardize_payment_methods()` | Map raw values to canonical form (e.g. 'paypal' → 'PayPal') |
| 5 | `standardize_transaction_status()` | Map raw statuses to canonical title-case |

> **Next step in pipeline**: `validation.ipynb`

In [ ]:
import sys
import os

# Add ml_engine/ to path so we can import our modules
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
from cleaning import (
    load_dataset,
    clean_dates,
    parse_prices,
    standardize_payment_methods,
    standardize_transaction_status,
    run_cleaning,
)

DATA_PATH = os.path.join("..", "data", "dataset_ecommerce_transactions_data.csv")
print(f"Data path : {os.path.abspath(DATA_PATH)}")

## 1. Load Raw Dataset

In [ ]:
df_raw = load_dataset(DATA_PATH)
print(f"Shape    : {df_raw.shape}")
print(f"Columns  : {df_raw.columns.tolist()}")
print(f"\nData types (raw):")
print(df_raw.dtypes)
df_raw.head(3)

## 2. Run Full Cleaning Pipeline

In [ ]:
df_clean = run_cleaning(df_raw, impute=False)

print("=== Cleaning Summary ===")
print(f"  Input rows  : {len(df_raw):,}")
print(f"  Output rows : {len(df_clean):,}")
df_clean.head(3)

## 3. Before vs After — Data Types

In [ ]:
dtype_comparison = pd.DataFrame({
    "Before" : df_raw.dtypes.astype(str),
    "After"  : df_clean.dtypes.astype(str),
})
print(dtype_comparison)

## 4. Standardised Value Checks

In [ ]:
print("=== Payment_Method after cleaning ===")
print(df_clean["Payment_Method"].value_counts())

print("\n=== Transaction_Status after cleaning ===")
print(df_clean["Transaction_Status"].value_counts())

## 5. Missing Values After Cleaning

In [ ]:
missing = pd.DataFrame({
    "Missing Count" : df_clean.isnull().sum(),
    "Missing %"     : (df_clean.isnull().mean() * 100).round(2),
})
print(missing[missing["Missing Count"] > 0])

---
## Key Takeaways

- `Transaction_Date` is now `datetime64` — unparseable values become `NaT` (visible in Missing Values)
- `Price` is now `float64` — currency symbols stripped
- `Payment_Method` and `Transaction_Status` are mapped to canonical values
- Missing values are **preserved** (`impute=False`) so validation can accurately count them
- Use `run_cleaning(df, impute=True)` to also fill numeric columns with their column median